# Emotion Recognition with Enhanced Att-1DCNN-GRU + TACO Cross-Attention

**Paper:** "Emotion recognition based on multimodal physiological electrical signals"

**Dataset:** DREAMER (23 subjects, 18 trials, 14-ch EEG @ 128Hz, 2-ch ECG @ 256Hz)

This notebook clones the full repository and imports all modules directly — no code is duplicated.

## 0. Setup

Run **cell A** the first time. If the session crashes and restarts, skip to **cell B** to reconnect without re-cloning.

In [ ]:
# ── Cell A: First-time setup (clone + install) ───────────────────────────────
REPO_URL = 'https://github.com/Sisoodiya/Sanjay-sWork.git'  # <-- CHANGE THIS
REPO_DIR = '/content/emotion-recognition'

import os, sys

# Install Git LFS and clone (DREAMER.mat is pulled via LFS)
!sudo apt-get install -qq git-lfs
!git lfs install
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install dependencies + CuPy for GPU-accelerated preprocessing
!pip install -q -r requirements.txt
!pip install -q cupy-cuda12x

# Verify dataset
assert os.path.exists('data/DREAMER.mat'), 'DREAMER.mat not found! Check REPO_URL.'
file_size = os.path.getsize('data/DREAMER.mat') / (1024 * 1024)
if file_size < 1:
    print('LFS pointer detected, pulling actual data...')
    !git lfs pull
    file_size = os.path.getsize('data/DREAMER.mat') / (1024 * 1024)
print(f'Ready: DREAMER.mat ({file_size:.0f} MB)')

In [ ]:
# ── Cell B: Reconnect after crash (skip Cell A if repo already cloned) ────────
import os, sys
REPO_DIR = '/content/emotion-recognition'

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Reconnected: {os.getcwd()}')

## 1. Imports

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src import config
from src.utils import set_seed, HAS_CUPY
from src.preprocessing import build_dataset
from src.transforms import ecg_to_2d, eeg_to_2d_grid
from src.model import build_model
from src.evaluate import print_results_table
from train import losocv_train

set_seed(config.RANDOM_SEED)

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print(f'CuPy acceleration: {"enabled" if HAS_CUPY else "disabled (falling back to NumPy)"}')

## 2. Load & Preprocess Data

In [ ]:
print('Building preprocessed dataset (filtering, ICA, segmentation)...')
dataset = build_dataset()

total = sum(d['eeg_segments'].shape[0] for d in dataset)
print(f'\nTotal segments: {total} '
      f'(expected ~{config.NUM_SUBJECTS * config.NUM_TRIALS * config.LAST_SECONDS})')

## 3. Visualize 2D Transforms

In [ ]:
# ECG: GAF, Recurrence Plot, Markov Transition Field (2 channels x 3 transforms = 6)
sample_2d = ecg_to_2d(dataset[0]['ecg_segments'][0])

titles = ['Ch1 GAF', 'Ch1 RP', 'Ch1 MTF', 'Ch2 GAF', 'Ch2 RP', 'Ch2 MTF']
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for idx, ax in enumerate(axes.flat):
    ax.imshow(sample_2d[:, :, idx], cmap='viridis', aspect='auto')
    ax.set_title(titles[idx])
    ax.axis('off')
plt.suptitle('ECG 2D Transforms (Subject 0, Segment 0)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# EEG: 14 channels mapped onto 9x9 spatial grid
sample_grid = eeg_to_2d_grid(dataset[0]['eeg_segments'][0])

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(sample_grid[0], cmap='RdBu_r', aspect='auto')
ax.set_title('EEG 9x9 Spatial Grid (t=0)')
for ch_idx, (row, col) in config.EEG_GRID_MAP.items():
    ax.text(col, row, config.EEG_ELECTRODE_NAMES[ch_idx], ha='center', va='center',
            fontsize=7, color='black', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Model Summary

In [ ]:
model = build_model()
model.summary()

# Quick forward pass sanity check
test_eeg = np.random.randn(2, 128, 9, 9).astype(np.float32)
test_ecg = np.random.randn(2, 64, 64, 6).astype(np.float32)
out = model.predict([test_eeg, test_ecg], verbose=0)
print(f'\nForward pass OK: {test_eeg.shape} + {test_ecg.shape} -> {out.shape}')

del model
tf.keras.backend.clear_session()

## 5. LOSOCV Training

Leave-One-Subject-Out Cross-Validation using `losocv_train()` from `train.py`.

### Quick Test (2 Subjects)

In [ ]:
quick_results = {}
for target in config.TARGETS:
    print(f"\n{'#' * 70}")
    print(f'  TRAINING: {target.upper()}')
    print(f"{'#' * 70}")
    quick_results[target] = losocv_train(dataset, target=target, num_subjects=2)

print_results_table(quick_results)

### Full LOSOCV (All 23 Subjects)

Uncomment to run the complete experiment. Takes several hours on GPU.

In [ ]:
# full_results = {}
# for target in config.TARGETS:
#     print(f"\n{'#' * 70}")
#     print(f'  TRAINING: {target.upper()}')
#     print(f"{'#' * 70}")
#     full_results[target] = losocv_train(dataset, target=target)
#
# print_results_table(full_results)